In [11]:
import pandas as pd
import numpy as np
import os

# Chargement des données brutes
df = pd.read_csv('../data/raw/data.csv') # Remplace par data.xlsx et pd.read_excel si ton fichier brut est un Excel
print(f"Dimensions initiales : {df.shape}")

Dimensions initiales : (4372, 52)


In [12]:
# 1. Suppression de la colonne constante
if 'NewsletterSubscribed' in df.columns:
    df = df.drop(columns=['NewsletterSubscribed'])

# 2. Correction de MonetaryTotal (valeurs datetime parasites venant d'Excel)
def to_numeric_safe(val):
    if isinstance(val, pd.Timestamp):
        return np.nan
    try:
        return float(val)
    except (ValueError, TypeError):
        return np.nan

if "MonetaryTotal" in df.columns:
    df["MonetaryTotal"] = df["MonetaryTotal"].apply(to_numeric_safe)

# 3. Parsing de RegistrationDate (avec format='mixed' pour éviter l'avertissement)
if 'RegistrationDate' in df.columns:
    df['RegistrationDate'] = pd.to_datetime(df['RegistrationDate'], format='mixed', dayfirst=True, errors='coerce')
    df["RegYear"]    = df["RegistrationDate"].dt.year
    df["RegMonth"]   = df["RegistrationDate"].dt.month
    df["RegDay"]     = df["RegistrationDate"].dt.day
    df["RegWeekday"] = df["RegistrationDate"].dt.weekday
    df = df.drop(columns=["RegistrationDate"])

# 4. Uniformisation SANS minuscules 
free_text_cols = ["Country", "Gender", "AccountStatus"]
for col in free_text_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

print("Nettoyage brut terminé.")

Nettoyage brut terminé.


In [13]:
# 1. Feature engineering depuis l'adresse IP
if "LastLoginIP" in df.columns:
    def is_private(ip):
        try:
            parts = str(ip).split(".")
            if len(parts) != 4: return 0
            first, second = int(parts[0]), int(parts[1])
            return int(first == 10 or (first == 172 and 16 <= second <= 31) or (first == 192 and second == 168))
        except Exception:
            return 0

    def first_octet(ip):
        try:
            return int(str(ip).split(".")[0])
        except Exception:
            return -1

    df["IP_IsPrivate"]  = df["LastLoginIP"].apply(is_private)
    df["IP_FirstOctet"] = df["LastLoginIP"].apply(first_octet)
    df = df.drop(columns=["LastLoginIP"])

# 2. Création de features dérivées monétaires
if "MonetaryTotal" in df.columns and "Recency" in df.columns:
    df["MonetaryPerDay"] = df["MonetaryTotal"] / (df["Recency"] + 1)
if "MonetaryTotal" in df.columns and "Frequency" in df.columns:
    df["AvgBasketValue"] = df["MonetaryTotal"] / df["Frequency"].replace(0, np.nan)
if "Recency" in df.columns and "CustomerTenureDays" in df.columns:
    df["TenureRatio"] = df["Recency"] / (df["CustomerTenureDays"] + 1)

print(f"Feature engineering terminé. Colonnes actuelles : {df.shape[1]}")

Feature engineering terminé. Colonnes actuelles : 58


In [15]:
# Création du dossier s'il n'existe pas
os.makedirs('../data/processed', exist_ok=True)

# Sauvegarde pour le Notebook 02
df.to_csv('../data/processed/data_cleaned.csv', index=False)

print("Fichier '../data/processed/data_cleaned.csv' sauvegardé avec succès !")

Fichier '../data/processed/data_cleaned.csv' sauvegardé avec succès !
